# Investigating Genes found by the Neural Networks

### NCBI Blast then Gene Search

In [ ]:
from paths import path_to_nn_runs
import os


In [ ]:
# Read nn_runs
nn_run = "inv_attr_genes_run7"
# if nn_run not in os.listdir(path_to_nn_runs):
#     raise ValueError("No run found")

def extract_kmer_line(file_path):
    # Check if filepath exists
    try:
        os.path.exists(file_path)
    except FileExistsError as e:
        print("File path doesn't exist")
    
    with open(file_path, "r") as logfile:
        for line in logfile:
            if "Top 10 decoded kmers:" in line:
                return line

def clean_kmer_line(kmer_line):
    """Clean the line containing kmers, from a messy string with noise, to a list with only decoded kmers"""
    kmers_string = kmer_line.split(":")[-1].strip()
    return kmers_string.strip("[]").replace("'", "").split(", ")

kmer_line = extract_kmer_line(path_to_nn_runs+nn_run+"/log_run7.txt")
print(clean_kmer_line(kmer_line))

In [ ]:
import time
from Bio.Blast import NCBIWWW, NCBIXML
from Bio import Entrez, SeqIO

# 1. Configuration
Entrez.email = "s215045@student.dtu.dk" 
kmers = ['CACAGCAAGCAA', 'AGAGAAGAAAGT', 'AATCACTGTCAA', 'TTCGCGTCAGAA', 'ATGACATACCAT', 'GAACAATGAGCC', 'AAGTTGAATTTG', 'ATGAAGCTGGTT', 'GGGTAAATATCC', 'AAGAGTGCTTGA']

def search_and_annotate_kmers(kmer_list, outfile:str = root+"logs/NCBI_gene_search.txt", acc_num:int = 3, tax_origin:str  = "txid38018[orgn]", ncbi_program:str = "blastn", ncbi_db:str = "core_nt"):
    """
    Blasts each of the kmers against NCBI, for related species (accessions), then searches its genes for the kmer along with possible functionalities
    """
    with open(outfile, "w") as logfile:
        print(f"Starting BLAST for {len(kmer_list)} kmers against Viral Database...", file=logfile)
        
        # We combine kmers into one FASTA-style string to save API calls
        fasta_query = "\n".join([f">kmer_{i}\n{k}" for i, k in enumerate(kmer_list)])
        
        try:
            # qblast parameters for short sequences:
            # - program: blastn
            # - database: nt (nucleotide)
            # - entrez_query: Restrict to Viruses
            # - word_size: 7 (minimum for blastn)
            # - expect: 1000 (higher to catch short hits)
            result_handle = NCBIWWW.qblast(
                program=ncbi_program, 
                database=ncbi_db, 
                sequence=fasta_query,
                entrez_query=tax_origin, #12333
                word_size=7,
                expect=1000,
                short_query=True
            )
            
            blast_records = NCBIXML.parse(result_handle)
            
            for record in blast_records:
                kmer_seq = kmer_list[int(record.query.split('_')[1])]
                print(f"\n--- Results for Kmer: {kmer_seq} ---", file=logfile)
                
                if not record.alignments:
                    print("No significant phage hits found.", file=logfile)
                    continue

                # Check the top acc_num hits for functional relevance
                for alignment in record.alignments[:acc_num]:
                    accession = alignment.accession
                    hit_def = alignment.title
                    
                    # Fetch GenBank record to find the specific gene overlapping the hit
                    print(f"Checking Gene in Hit: {accession} ({hit_def[:50]}...)", file=logfile)
                    
                    # We fetch the specific region of the hit to save bandwidth
                    hsp = alignment.hsps[0]
                    start, end = min(hsp.sbjct_start, hsp.sbjct_end), max(hsp.sbjct_start, hsp.sbjct_end)
                    
                    try:
                        handle = Entrez.efetch(db="nucleotide", id=accession, rettype="gb", retmode="text")
                        genbank_rec = SeqIO.read(handle, "genbank")
                        handle.close()
                        
                        found_gene = False
                        for feature in genbank_rec.features:
                            if feature.type == "CDS":
                                # Check if the kmer location overlaps with this gene
                                if start >= feature.location.start and end <= feature.location.end:
                                    product = feature.qualifiers.get('product', ['Unknown'])[0]
                                    gene = feature.qualifiers.get('gene', ['N/A'])[0]
                                    print(f"  [MATCH] Found in Gene: {gene} | Function: {product}", file=logfile)
                                    found_gene = True
                                    break
                        if not found_gene:
                            print("  [INFO] Hit is in an intergenic/non-coding region.", file=logfile)
                            
                    except Exception as e:
                        print(f"  [ERROR] Could not fetch details for {accession}: {e}", file=logfile)
                    
                    time.sleep(1) # Be nice to NCBI servers

        except Exception as e:
            print(f"BLAST search failed: {e}", file=logfile)

# Run the search
search_and_annotate_kmers(kmers, path_to_nn_runs+nn_run+"NCBI_gene_search.txt")

### Read and plot results file

In [ ]:
import pandas as pd
blast_results_path = path_to_nn_runs+"torch_mlp_n400_k12_standard_run92/GA_kmers_blast_results.csv"
blast_results = pd.read_csv(blast_results_path)
print(blast_results.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.countplot(data=blast_results, x="Function", order=blast_results["Function"].value_counts().index)
plt.xticks(rotation=45, ha='right')
plt.title("Distribution of BLAST Hits by Organism")
plt.xlabel("Organism")
plt.ylabel("Count of Hits")
plt.tight_layout()
plt.show()

In [ ]:
organisms = sorted(blast_results["organism"].dropna().unique())
n_orgs = len(organisms)

fig, axes = plt.subplots(2, n_orgs, figsize=(7 * n_orgs, 12), sharey="row")
plt.title("Distribution of BLAST Hits by Function and Gene, per Organism", fontsize=16)

# Ensure axes is always 2D: [row][col]
if n_orgs == 1:
    axes = [[axes[0]], [axes[1]]]

for i, org in enumerate(organisms):
    # Row 1: Function
    ax_func = axes[0][i]
    subset_func = blast_results[(blast_results["organism"] == org) & (blast_results["Function"].notna())]
    order_func = subset_func["Function"].value_counts().index

    if subset_func.empty:
        ax_func.text(0.5, 0.5, "No Function data", ha="center", va="center", transform=ax_func.transAxes)
        ax_func.set_xticks([])
    else:
        sns.countplot(data=subset_func, x="Function", order=order_func, ax=ax_func)
        ax_func.tick_params(axis="x", rotation=45)
        for lbl in ax_func.get_xticklabels():
            lbl.set_ha("right")

    ax_func.set_title(f"{org} count of annotated Kmer Functions")
    ax_func.set_xlabel("Function")
    ax_func.set_ylabel("Count")

    # Row 2: Gene
    ax_gene = axes[1][i]
    subset_gene = blast_results[(blast_results["organism"] == org) & (blast_results["Gene"].notna())]
    order_gene = subset_gene["Gene"].value_counts().index

    if subset_gene.empty:
        ax_gene.text(0.5, 0.5, "No Gene data", ha="center", va="center", transform=ax_gene.transAxes)
        ax_gene.set_xticks([])
    else:
        sns.countplot(data=subset_gene, x="Gene", order=order_gene, ax=ax_gene)
        ax_gene.tick_params(axis="x", rotation=45)
        for lbl in ax_gene.get_xticklabels():
            lbl.set_ha("right")

    ax_gene.set_title(f"{org} count of annotated Kmer Genes")
    ax_gene.set_xlabel("Gene")
    ax_gene.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
plt.subplots(figsize=(10,6), )

# Circos plot of annotated genomes

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from pycirclize import Circos
from pycirclize.parser import Genbank
from paths import data_prod_path

def _best_feature_label(feature):
    qualifiers = feature.qualifiers
    for key in ("gene", "locus_tag", "product"):
        value = qualifiers.get(key, [""])[0] if qualifiers.get(key) else ""
        if isinstance(value, str) and value.strip():
            return value.strip()
    return ""

def _feature_midpoint(feature):
    return (int(feature.location.start) + int(feature.location.end)) / 2

def plot_circos_from_gbk(
    bact: str,
    gbk_path: str | Path | None = None,
    output_dir: str | Path | None = None,
    window_size: int = 5000,
    gene_labels: list[str] | None = None,
    label_top_n_genes: int | None = None,
    track_labels: list[str] | None = None,
    plot_sample: bool = False,
    random_seed: int = 42,
    figsize: tuple[float, float] = (10, 10),
    silent: bool = False
):
    if gbk_path is None:
        gbk_path = Path(data_prod_path) / "prokka_bacts" / bact / f"{bact}.gbk"
    else:
        gbk_path = Path(gbk_path)

    if not gbk_path.exists():
        raise FileNotFoundError(f"GenBank file not found: {gbk_path}")

    if output_dir is None:
        output_dir = Path.cwd()
    else:
        output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    gbk = Genbank(str(gbk_path))
    circos = Circos(gbk.get_seqid2size(), space=5)

    cds_plus_by_seqid = gbk.get_seqid2features(feature_type="CDS", target_strand=1)
    cds_minus_by_seqid = gbk.get_seqid2features(feature_type="CDS", target_strand=-1)
    cds_all_by_seqid = gbk.get_seqid2features(feature_type="CDS")
    trna_by_seqid = gbk.get_seqid2features(feature_type="tRNA")
    rrna_by_seqid = gbk.get_seqid2features(feature_type="rRNA")
    tmrna_by_seqid = gbk.get_seqid2features(feature_type="tmRNA")
    seq_map = gbk.get_seqid2seq()

    requested_labels = set(gene_labels) if gene_labels else set()
    labels_added = set()
    track_labels_set = set(track_labels or [])

    # Build label pool across all CDS features
    all_label_candidates = []
    seen = set()
    for seqid in cds_all_by_seqid:
        for feature in cds_all_by_seqid.get(seqid, []):
            label = _best_feature_label(feature)
            if label and label not in seen:
                all_label_candidates.append(label)
                seen.add(label)

    # Label-selection rule:
    # 1) explicit gene_labels if provided
    # 2) top N if label_top_n_genes is set
    # 3) random sampling of 1/40 labels if label_top_n_genes is None
    if requested_labels:
        selected_label_pool = list(requested_labels)
        if plot_sample:
            sample_size = max(1, len(all_label_candidates) // 40) if all_label_candidates else 0
            rng = np.random.default_rng(random_seed)
            sampled = rng.choice(all_label_candidates, size=sample_size, replace=False).tolist() if sample_size > 0 else []
            selected_label_pool.extend(sampled)
    elif label_top_n_genes is not None:
        selected_label_pool = set(all_label_candidates[:label_top_n_genes])
    else:
        sample_size = max(1, len(all_label_candidates) // 40) if all_label_candidates else 0
        rng = np.random.default_rng(random_seed)
        sampled = rng.choice(all_label_candidates, size=sample_size, replace=False).tolist() if sample_size > 0 else []
        selected_label_pool = set(sampled)

    first_sector_name = circos.sectors[0].name if circos.sectors else None

    for sector in circos.sectors:
        seqid = sector.name
        sector.axis(fc="#eeeeee", ec="none")
        center_x = (sector.start + sector.end) / 2

        # Track 1: Forward-strand CDS
        f_track = sector.add_track((92, 97), name="Forward CDS")
        f_feats = cds_plus_by_seqid.get(seqid, [])
        if f_feats:
            f_track.genomic_features(f_feats, color="salmon", lw=0.3)

        # Track 2: Reverse-strand CDS
        r_track = sector.add_track((87, 92), name="Reverse CDS")
        r_feats = cds_minus_by_seqid.get(seqid, [])
        if r_feats:
            r_track.genomic_features(r_feats, color="skyblue", lw=0.3)

        # Track 3: RNA features
        rna_track = sector.add_track((82, 85), name="RNAs")
        rna_feats = [
            *trna_by_seqid.get(seqid, []),
            *rrna_by_seqid.get(seqid, []),
            *tmrna_by_seqid.get(seqid, []),
        ]
        if rna_feats:
            rna_track.genomic_features(rna_feats, color="black", lw=1)

        # Track 4: GC content
        gc_track = sector.add_track((68, 78), name="GC Content")
        seq_text = seq_map.get(seqid, "")
        pos_gc, content = gbk.calc_gc_content(window_size=window_size, seq=seq_text)
        gc_min = float(np.min(content)) if len(content) else 0.0
        gc_max = float(np.max(content)) if len(content) else 1.0
        gc_mid = (gc_min + gc_max) / 2.0
        gc_hi = np.where(content > gc_mid, content, gc_mid)
        gc_lo = np.where(content < gc_mid, content, gc_mid)
        gc_track.line(pos_gc, content, color="black", lw=0.5)
        gc_track.fill_between(pos_gc, gc_hi, gc_mid, vmin=gc_min, vmax=gc_max, color="green", alpha=0.4)
        gc_track.fill_between(pos_gc, gc_lo, gc_mid, vmin=gc_min, vmax=gc_max, color="red", alpha=0.4)

        # Track 5: GC skew
        skew_track = sector.add_track((56, 66), name="GC Skew")
        pos_skew, skew = gbk.calc_gc_skew(window_size=window_size, seq=seq_text)
        max_abs_skew = float(np.max(np.abs(skew))) if len(skew) else 1.0
        skew_pos = np.where(skew > 0, skew, 0.0)
        skew_neg = np.where(skew < 0, skew, 0.0)
        skew_track.fill_between(
            pos_skew,
            skew_pos,
            0.0,
            vmin=-max_abs_skew,
            vmax=max_abs_skew,
            color="purple",
            alpha=0.4,
        )
        skew_track.fill_between(
            pos_skew,
            skew_neg,
            0.0,
            vmin=-max_abs_skew,
            vmax=max_abs_skew,
            color="orange",
            alpha=0.4,
        )

        # Outer label track: keep all text outside plotted data tracks
        outer_label_track = sector.add_track((104, 126), name="Outer Labels")

        # Gene labels/markings for identification (outside tracks)
        cds_feats = cds_all_by_seqid.get(seqid, [])
        label_candidates = []
        for feature in cds_feats:
            label = _best_feature_label(feature)
            if not label or label in labels_added:
                continue
            if label in selected_label_pool:
                label_candidates.append((feature, label))

        for feature, label in label_candidates:
            midpoint = _feature_midpoint(feature)
            outer_label_track.annotate(
                midpoint,
                label,
                min_r=105,
                max_r=124,
                label_size=7,
                shorten=24,
                line_kws={"color": "dimgray", "lw": 0.7},
                text_kws={"color": "black"},
            )
            labels_added.add(label)

        # Optional track-name labels, also outside tracks
        if seqid == first_sector_name:
            if "forward_cds" in track_labels_set:
                outer_label_track.text("Forward CDS", x=center_x, r=118, size=8, color="salmon")
            if "reverse_cds" in track_labels_set:
                outer_label_track.text("Reverse CDS", x=center_x, r=116, size=8, color="deepskyblue")
            if "rna" in track_labels_set:
                outer_label_track.text("RNA", x=center_x, r=114, size=8, color="black")
            if "gene_labels" in track_labels_set:
                outer_label_track.text("Gene labels", x=center_x, r=112, size=8, color="dimgray")
            if "gc_content" in track_labels_set:
                outer_label_track.text("GC content", x=center_x, r=110, size=8, color="black")
            if "gc_skew" in track_labels_set:
                outer_label_track.text("GC skew", x=center_x, r=108, size=8, color="purple")

    fig = circos.plotfig(figsize=figsize)
    fig.suptitle(f"Circos Genome Map: {bact}", fontsize=14, y=0.98)

    legend_handles = [
        mpatches.Patch(color="salmon", label="CDS (+ strand)"),
        mpatches.Patch(color="skyblue", label="CDS (- strand)"),
        mpatches.Patch(color="black", label="RNA genes (tRNA/rRNA/tmRNA)"),
        mlines.Line2D([], [], color="dimgray", lw=1, label="Annotated gene label marker"),
        mpatches.Patch(color="green", alpha=0.5, label="GC content above midpoint"),
        mpatches.Patch(color="red", alpha=0.5, label="GC content below midpoint"),
        mpatches.Patch(color="purple", alpha=0.5, label="GC skew positive"),
        mpatches.Patch(color="orange", alpha=0.5, label="GC skew negative"),
    ]
    ax = fig.axes[0]
    ax.legend(
        handles=legend_handles,
        loc="upper right",
        bbox_to_anchor=(0, 0),
        fontsize=8,
        frameon=False,
    )

    output_png = output_dir / f"{bact}_genome_map.png"
    fig.savefig(output_png, dpi=300, bbox_inches="tight")
    if not silent: print(f"Saved Circos plot to: {output_png}")

    return fig, output_png

# Example run
bact = "J105_22_reoriented_merged"
fig, output_png = plot_circos_from_gbk(
    bact = bact,
    output_dir = f"{Path(data_prod_path)}/prokka_bacts/{bact}/",
    gene_labels = [
    "cas1", "cas2-3", "csy1", "csy2", "csy3", "csy4", 
    "mazE", "mazF", "relB", "relE", "hicA", "hicB", 
    "hipA", "hipB", "parE", "hsdS", "hsdM", "hsdR", 
    "dndC", "dndD"],
    plot_sample=True, 
    silent = True
)

### In batch for all annotated bacterial genomes

In [ ]:
import os
from tqdm import tqdm
for dir in tqdm(os.listdir(data_prod_path+"/prokka_bacts/")):
    bact = dir.split("_reoriented_merged")[0]
    inner_dir = os.path.join(data_prod_path, "prokka_bacts", dir)
    for file in os.listdir(inner_dir):
        if file.endswith(".gbk"):
            gbk_path = os.path.join(inner_dir, file)
            output_dir = inner_dir
            #print(f"Processing {bact}...")
            #print(f"GBK path: {gbk_path}")
            plot_circos_from_gbk(
                bact=bact,
                gbk_path=gbk_path,
                output_dir=output_dir,
                gene_labels=[
                    "cas1", "cas2-3", "csy1", "csy2", "csy3", "csy4", 
                    "mazE", "mazF", "relB", "relE", "hicA", "hicB", 
                    "hipA", "hipB", "parE", "hsdS", "hsdM", "hsdR", 
                    "dndC", "dndD"],
                plot_sample=True
            )

# Mapping PFI values to Kmers in Genes
1) read and locate high/low pfi value hash values
2) decode using KmerCodecs (not available for sourmash)


In [ ]:
from pathlib import Path
import pandas as pd
from Bio import SeqIO
from paths import data_prod_path

ANNOTATED_SNIPPET_ROOT = Path(data_prod_path) / "annotated_snippet"

def _normalize_kmer(kmer: str) -> str:
    return str(kmer).strip().upper()

def extract_bacteria_genes_for_kmer(kmer: str, strain_name: str, root_dir: str | Path = ANNOTATED_SNIPPET_ROOT) -> pd.DataFrame:
    """Return bacterial gene annotations for records whose sequence contains `kmer`.

    Searches for:
    - a file ending in `_merged.ffn`
    - a companion file ending in `_merged.tsv`

    The matching record IDs from the FASTA headers are matched against the
    `locus_tag` column in the TSV file.
    """
    root_dir = Path(root_dir)
    kmer = _normalize_kmer(kmer)

    strain_dirs = [p for p in (root_dir / "prokka_bacts").rglob("*") if p.is_dir() and strain_name in p.name]
    if not strain_dirs:
        raise FileNotFoundError(f"No bacteria directory found for strain '{strain_name}' under {root_dir / 'prokka_bacts'}")

    strain_dir = strain_dirs[0]
    ffn_files = sorted(strain_dir.glob("*_merged.ffn"))
    tsv_files = sorted(strain_dir.glob("*_merged.tsv"))
    if not ffn_files:
        raise FileNotFoundError(f"No *_merged.ffn file found in {strain_dir}")
    if not tsv_files:
        raise FileNotFoundError(f"No *_merged.tsv file found in {strain_dir}")

    matching_locus_tags = []
    for record in SeqIO.parse(str(ffn_files[0]), "fasta"):
        if kmer in str(record.seq).upper():
            matching_locus_tags.append(record.id)

    if not matching_locus_tags:
        return pd.DataFrame(columns=["locus_tag", "length_bp", "gene", "product"])

    ann_df = pd.read_csv(tsv_files[0], sep="\t")
    cols = ["locus_tag", "length_bp", "gene", "product"]
    missing = [c for c in cols if c not in ann_df.columns]
    if missing:
        raise KeyError(f"Missing expected columns in {tsv_files[0]}: {missing}")

    result = ann_df[ann_df["locus_tag"].astype(str).isin(matching_locus_tags)][cols].copy()
    return result.reset_index(drop=True)

def extract_phage_genes_for_kmer(kmer: str, strain_name: str, root_dir: str | Path = ANNOTATED_SNIPPET_ROOT) -> pd.DataFrame:
    """Return phage gene annotations for records whose sequence contains `kmer`.

    Searches for:
    - a `phanotate.ffn` file under the pharokka results for the strain
    - a `*_per_cds_predictions.tsv` file under the phold results for the strain

    The matching FASTA record IDs are matched against the `cds_id` column in the
    PHOLD table.
    """
    root_dir = Path(root_dir)
    kmer = _normalize_kmer(kmer)

    pharokka_root = root_dir / "pharokka"
    phold_root = root_dir / "phold"

    pharokka_dirs = [p for p in pharokka_root.rglob("*") if p.is_dir() and strain_name in p.name]
    phold_dirs = [p for p in phold_root.rglob("*") if p.is_dir() and strain_name in p.name]
    if not pharokka_dirs:
        raise FileNotFoundError(f"No pharokka directory found for strain '{strain_name}' under {pharokka_root}")
    if not phold_dirs:
        raise FileNotFoundError(f"No phold directory found for strain '{strain_name}' under {phold_root}")

    pharokka_dir = pharokka_dirs[0]
    phold_dir = phold_dirs[0]

    ffn_files = sorted(pharokka_dir.glob("**/phanotate.ffn"))
    if not ffn_files:
        raise FileNotFoundError(f"No phanotate.ffn file found in {pharokka_dir}")

    tsv_files = sorted(phold_dir.glob("**/*_per_cds_predictions.tsv"))
    if not tsv_files:
        raise FileNotFoundError(f"No *_per_cds_predictions.tsv file found in {phold_dir}")

    matching_cds_ids = []
    for record in SeqIO.parse(str(ffn_files[0]), "fasta"):
        if kmer in str(record.seq).upper():
            matching_cds_ids.append(record.id)

    if not matching_cds_ids:
        return pd.DataFrame(columns=[
            "contig_id", "cds_id", "start", "end", "phrog", "function", "product",
            "annotation_method", "annotation_confidence", "tophit_protein",
            "function_with_highest_bitscore_proportion", "prostt5_confidence"
        ])

    ann_df = pd.read_csv(tsv_files[0], sep="\t")
    cols = [
        "contig_id", "cds_id", "start", "end", "phrog", "function", "product",
        "annotation_method", "annotation_confidence", "tophit_protein",
        "function_with_highest_bitscore_proportion", "prostt5_confidence"
    ]
    missing = [c for c in cols if c not in ann_df.columns]
    if missing:
        raise KeyError(f"Missing expected columns in {tsv_files[0]}: {missing}")

    result = ann_df[ann_df["cds_id"].astype(str).isin(matching_cds_ids)][cols].copy()
    return result.reset_index(drop=True)


In [6]:
extract_bacteria_genes_for_kmer("CACAGCAAGCAA", "J76_21_reoriented_merged")

,locus_tag,length_bp,gene,product
0,AHHNCAAL_03872,390,tusD,NaN
1,AHHNCAAL_03872,390,tusD,Sulfurtransferase TusD


In [9]:
extract_phage_genes_for_kmer("CACAA", "Lelliottia_phage_Pantea")

,contig_id,cds_id,start,end,phrog,function,product,annotation_method,annotation_confidence,tophit_protein,function_with_highest_bitscore_proportion,prostt5_confidence
0,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0001,186,1,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,51.90625
1,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0004,686,414,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,45.37500
2,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0008,1674,1387,30110,unknown function,hypothetical protein,foldseek,high,envhog_8JHlt,unknown function,62.37500
3,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0010,2115,1933,30457,unknown function,hypothetical protein,foldseek,high,protein104025,unknown function,81.25000
4,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0012,2782,2354,14785,unknown function,hypothetical protein,foldseek,high,protein381140,unknown function,43.90625
...,...,...,...,...,...,...,...,...,...,...,...,...
129,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0275,143725,143961,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,50.96875
130,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0279,144591,144932,7366,unknown function,hypothetical protein,pharokka,pharokka,NaN,NaN,36.43750
131,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0292,148417,148782,60005,unknown function,hypothetical protein,foldseek,high,singleton27042,unknown function,43.75000
132,Lelliottia_phage_Pantea,QEWVWKBE_CDS_0294,149078,149218,No_PHROG,unknown function,hypothetical protein,none,none,NaN,NaN,48.81250
